In [ ]:
import os
MONGO_URI = os.environ["MONGO_URI"]


In [ ]:
import os
CLAUDE_API_KEY = os.environ["ANTHROPIC_API_KEY"]


In [ ]:
import os
# =========================================================
# IMPORTS
# =========================================================

from pymongo import MongoClient
from sentence_transformers import SentenceTransformer
from bson import ObjectId

import json
import traceback

# =========================================================
# CONFIG
# =========================================================

MONGO_URI = os.environ["MONGO_URI"]

DB_NAME = "365"

SOURCE_COLLECTION = "hotel"

EMBED_COLLECTION = "embedding6"

# MULTILINGUAL MODEL
EMBED_MODEL = "intfloat/multilingual-e5-base"

BATCH_SIZE = 8

# =========================================================
# CONNECT MONGODB
# =========================================================

print("Connecting MongoDB...")

client = MongoClient(
    MONGO_URI,
    serverSelectionTimeoutMS=30000
)

db = client[DB_NAME]

source_collection = db[SOURCE_COLLECTION]

embedding_collection = db[EMBED_COLLECTION]

print("MongoDB connected.")

# =========================================================
# CLEAR OLD EMBEDDINGS
# =========================================================

embedding_collection.delete_many({})

print("Old embeddings deleted.")

# =========================================================
# LOAD EMBEDDING MODEL
# =========================================================

print("Loading embedding model...")

embedder = SentenceTransformer(
    EMBED_MODEL
)

print("Embedding model loaded.")

# =========================================================
# CLEAN MONGO DATA
# =========================================================

def clean_mongo_data(data):

    if isinstance(data, dict):

        cleaned = {}

        for k, v in data.items():

            if k == "_id":
                continue

            cleaned[k] = clean_mongo_data(v)

        return cleaned

    elif isinstance(data, list):

        return [
            clean_mongo_data(x)
            for x in data
        ]

    elif isinstance(data, ObjectId):

        return str(data)

    else:

        return data

# =========================================================
# SAFE JSON
# =========================================================

def safe_json(data):

    return json.dumps(
        data,
        ensure_ascii=False,
        indent=2,
        default=str
    )

# =========================================================
# E5 FORMAT
# =========================================================

def format_document_text(text):

    # E5 requires "passage:"
    return f"passage: {text}"

# =========================================================
# BUILD OVERVIEW CHUNK
# =========================================================

def build_overview_chunk(hotel_doc):

    hotel_info = hotel_doc.get(
        "hotel_information",
        {}
    )

    hotel_name = hotel_info.get(
        "hotel_name",
        ""
    )

    room_names = [

        room.get(
            "room_name",
            ""
        )

        for room in hotel_doc.get(
            "room_types",
            []
        )
    ]

    facilities = [

        facility.get(
            "facility_name",
            ""
        )

        for facility in hotel_doc.get(
            "facilities",
            []
        )
    ]

    overview_text = f"""
Hotel name: {hotel_name}

Address:
{hotel_info.get("address", "")}

District:
{hotel_info.get("district", "")}

City:
{hotel_info.get("city", "")}

Country:
{hotel_info.get("country", "")}

Website:
{hotel_info.get("website", "")}

Available room types:
{", ".join(room_names)}

Facilities:
{", ".join(facilities)}

This hotel contains:
room rates,
meal information,
child policy,
cancellation policy,
checkin checkout policy,
payment policy,
extra bed policy,
transportation services,
facilities,
and hotel contract information.
"""

    return {

        "chunk_type":
            "hotel_overview",

        "hotel_name":
            hotel_name,

        "text":
            overview_text,

        "data": {

            "hotel_information":
                hotel_info
        }
    }

# =========================================================
# BUILD CHUNKS
# =========================================================

def build_chunks(hotel_doc):

    chunks = []

    hotel_doc = clean_mongo_data(
        hotel_doc
    )

    hotel_info = hotel_doc.get(
        "hotel_information",
        {}
    )

    hotel_name = hotel_info.get(
        "hotel_name",
        ""
    )

    # =====================================================
    # OVERVIEW CHUNK
    # =====================================================

    chunks.append(
        build_overview_chunk(
            hotel_doc
        )
    )

    # =====================================================
    # TOP LEVEL FIELDS
    # =====================================================

    top_level_fields = [

        "hotel_information",

        "contract_information",

        "room_types",

        "extra_bed_policy",

        "child_policy",

        "meal_information",

        "included_benefits",

        "facilities",

        "transportation_services",

        "checkin_checkout_policy",

        "cancellation_policy",

        "payment_policy",

        "peak_period_surcharge",

        "restrictions",

        "important_notes",

        "raw_season_definitions",

        "special_hotel_feature",

        "metadata"
    ]

    # =====================================================
    # 1 TOP LEVEL FIELD = 1 CHUNK
    # =====================================================

    for field in top_level_fields:

        field_data = hotel_doc.get(
            field,
            None
        )

        if field_data is None:
            continue

        chunk_text = f"""
Hotel:
{hotel_name}

Section:
{field}

JSON Content:
{safe_json(field_data)}
"""

        chunks.append({

            "chunk_type":
                field,

            "hotel_name":
                hotel_name,

            "text":
                chunk_text,

            "data": {
                field: field_data
            }
        })

    return chunks

# =========================================================
# START PROCESSING
# =========================================================

print("\n================================================")
print("START EMBEDDING")
print("================================================")

total_chunks = 0

success_hotels = 0

failed_hotels = 0

# =========================================================
# CURSOR
# =========================================================

cursor = source_collection.find({}).batch_size(10)

# =========================================================
# LOOP HOTELS
# =========================================================

for hotel_doc in cursor:

    try:

        hotel_doc = clean_mongo_data(
            hotel_doc
        )

        hotel_name = hotel_doc.get(
            "hotel_information",
            {}
        ).get(
            "hotel_name",
            "UNKNOWN"
        )

        mongo_id = str(
            hotel_doc.get(
                "_id",
                ""
            )
        )

        print("\n================================================")
        print(f"Processing: {hotel_name}")
        print(f"Mongo ID : {mongo_id}")
        print("================================================")

        # =================================================
        # BUILD CHUNKS
        # =================================================

        chunks = build_chunks(
            hotel_doc
        )

        print(
            f"Generated {len(chunks)} chunks"
        )

        # =================================================
        # EMBEDDING TEXTS
        # =================================================

        texts = []

        for chunk in chunks:

            formatted_text = format_document_text(
                chunk["text"]
            )

            texts.append(
                formatted_text
            )

        # =================================================
        # CREATE EMBEDDINGS
        # =================================================

        embeddings = embedder.encode(

            texts,

            batch_size=BATCH_SIZE,

            show_progress_bar=False,

            normalize_embeddings=True,

            convert_to_numpy=True
        )

        # =================================================
        # SAVE DOCUMENTS
        # =================================================

        save_docs = []

        for chunk, embedding in zip(
            chunks,
            embeddings
        ):

            save_doc = {

                "source_hotel_id":
                    mongo_id,

                "hotel_name":
                    chunk.get(
                        "hotel_name",
                        ""
                    ),

                "chunk_type":
                    chunk.get(
                        "chunk_type",
                        ""
                    ),

                "text":
                    chunk["text"],

                "data":
                    chunk["data"],

                "embedding":
                    embedding.tolist()
            }

            save_docs.append(
                save_doc
            )

        # =================================================
        # BULK INSERT
        # =================================================

        if save_docs:

            embedding_collection.insert_many(
                save_docs
            )

            total_chunks += len(
                save_docs
            )

        success_hotels += 1

        print(
            f"SUCCESS -> {hotel_name}"
        )

    except Exception as e:

        failed_hotels += 1

        print("\n================================================")
        print("ERROR PROCESSING HOTEL")
        print(f"Hotel: {hotel_name}")
        print(f"Mongo ID: {mongo_id}")
        print(f"Error Type: {type(e).__name__}")
        print(f"Error: {str(e)}")
        print("================================================")

        traceback.print_exc()

        continue

# =========================================================
# FINAL SUMMARY
# =========================================================

print("\n================================================")
print("DONE")
print("================================================")

print(
    f"Success hotels : "
    f"{success_hotels}"
)

print(
    f"Failed hotels  : "
    f"{failed_hotels}"
)

print(
    f"Total chunks   : "
    f"{total_chunks}"
)

print(
    f"Embedding collection count: "
    f"{embedding_collection.count_documents({})}"
)